In [56]:
import os
import pandas as pd
import glob

def read_csv_files(directory):
    # Create a list to hold DataFrames
    dataframes = []
    # Use glob to find all csv files in the specified directory
    csv_files = glob.glob(os.path.join(directory, "*.csv"))
    # Loop through each file and read it into a DataFrame
    for file in csv_files:
        df = pd.read_csv(file)
        dataframes.append(df)
    return dataframes

# Specify the directory containing the CSV files
directory = "sentiment-outputs-llm/processed_outputs/"
# Read the CSV files
sentiment_dataframes = read_csv_files(directory)

In [57]:
finance_data = pd.read_csv("data/final_dataset.csv")
finance_data

,Date,Disney,Microsoft,Walmart,Google,Exxon Mobil,Apple,Intel,JP Morgan,Johnson & Johnson,...,NASDAQ Composite,Dow Jones,S&P 500,Fed Funds Rate,Oil,Gold,10Y Treasury,EUR-USD,USD-JPY,news_text
0,2004-08-19,17.806047,16.771729,12.069075,2.496041,22.676308,0.461482,12.699708,21.737772,31.452284,...,1819.890015,10040.820312,1091.229980,1.4420,48.700001,407.100006,4.211,1.237195,109.360001,Democrats' Legal Challenges Impede Nader Campa...
1,2004-08-20,17.837843,16.821201,12.022876,2.694301,22.756765,0.462834,12.474672,22.109600,31.596300,...,1838.020020,10110.139648,1098.349976,1.4540,47.860001,413.200012,4.231,1.232195,109.110001,7 More Managers Fired Over Nortel Accounting W...
2,2004-08-23,17.623217,16.895638,11.835877,2.721417,22.626007,0.467042,12.630469,22.052399,31.601843,...,1838.699951,10073.049805,1095.680054,1.4770,46.049999,410.500000,4.279,1.215200,109.739998,World Briefing | Africa: Uganda: 68 Percent In...
3,2004-08-24,17.734505,16.895638,11.849078,2.608728,22.555599,0.480116,12.503527,22.075277,31.668293,...,1836.890015,10098.629883,1096.189941,1.5170,45.209999,403.000000,4.283,1.208196,109.690002,National Briefing Names of the Dead Metro Brie...
4,2004-08-25,17.885534,17.087910,11.901878,2.636839,22.726597,0.496646,12.665086,22.561522,31.939745,...,1860.719971,10181.740234,1104.959961,1.5120,43.470001,407.899994,4.261,1.208605,110.099998,9/11 Panel Leader Has Praise for Plan to Split...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5309,2024-12-25,112.555000,437.039734,92.257973,195.393150,104.538502,257.987671,20.420000,239.999573,143.329117,...,20025.745117,43311.416016,6038.814941,4.2075,69.860001,2629.400024,4.585,1.040258,157.106995,Judge Strikes Down Portions of Arkansas Law Th...
5310,2024-12-26,112.550003,436.432068,92.312691,195.138748,104.582695,258.396667,20.440001,240.409912,143.196320,...,20020.359375,43325.800781,6037.589844,4.2150,69.620003,2638.800049,4.579,1.039955,157.132996,Thursday Briefing: Rebel Factions Try to Unite...
5311,2024-12-27,111.550003,428.881104,91.188515,192.305435,104.572884,254.974930,20.299999,238.462036,142.675003,...,19722.029297,42992.210938,5970.839844,4.1780,70.599998,2617.199951,4.619,1.042318,157.748001,Friday Briefing: How Israel Weakened Civilian ...
5312,2024-12-30,110.800003,423.202911,90.104111,190.789047,103.865776,251.593094,19.820000,236.632812,140.992996,...,19486.789062,42573.730469,5906.939941,4.1820,70.989998,2606.100098,4.545,1.042938,157.873001,The Evening: Record Homelessness in the U.S. Y...


In [58]:
merged_dataframes = []

for i in range(len(sentiment_dataframes)):
    sentiment_dataframes[i].rename(columns={'date': 'Date'}, inplace=True)
    merged_data = pd.merge(finance_data, sentiment_dataframes[i], on='Date', how='inner')
    merged_dataframes.append(merged_data)


In [59]:
stocks_in_order =['10Y Treasury','Apple','Coca-Cola','Disney','Dow Jones',
                'EUR-USD','Exxon Mobil','Fed Funds Rate','Gold','Google',
                'Intel','Johnson & Johnson','JP Morgan','Microsoft','NASDAQ Composite',
                'Oil','S&P 500','USD-JPY','Walmart']

In [60]:
j = 0
for merged_data in merged_dataframes:
    stock = stocks_in_order[j]
    print(stock)
    data = pd.DataFrame()
    data['Date'] = merged_data['Date']
    # Create 'Day 1-10' columns
    for i in range(1, 11):
        # Change disney to appropriate stock name
        data[f'Day {i}'] = (merged_data[stock].shift(i) - merged_data[stock].shift(i-1)) / merged_data[stock].shift(i-1)
    # Create 'sentiment' column
    data['mean_score'] = merged_data['mean_score']
    data['mean_class'] = merged_data['mean_class']
    data['majority_vote_score'] = merged_data['majority_vote_score']
    data['median_score'] = merged_data['median_score']

    data.dropna(inplace=True)
    data.reset_index(drop=True, inplace=True)

    df = data.copy()

    price_columns = [f'Day {i}' for i in range(1, 11)]

    # Get the day 1 of the row below for next_return calculationrn
    df['next_return'] = df['Day 1'].shift(-1)

    # Drop rows where next_return is NaN (last row)
    df = df[:-1]

    # Create movement label (classification target)
    def get_movement_label(r):
        if r > 0.005:
            return 2
        elif r < -0.005:
            return 0
        else:
            return 1

    df['movement_label'] = df['next_return'].apply(get_movement_label)

    # Final columns: Day1-Day10, Sentiment, movement_label / next_return
    classification_df = df[price_columns + ['mean_score','mean_class','majority_vote_score','median_score', 'movement_label', 'Date']]
    regression_df = df[price_columns + ['mean_score','mean_class','majority_vote_score','median_score', 'next_return' , 'Date']]

    classification_df.to_csv(f"LSTM Models/data/classification_data_returns_{stock}.csv", index=False)
    regression_df.to_csv(f"LSTM Models/data/regression_data_returns_{stock}.csv", index=False)

    j = j + 1

10Y Treasury
Apple
Coca-Cola
Disney
Dow Jones
EUR-USD
Exxon Mobil
Fed Funds Rate
Gold
Google
Intel
Johnson & Johnson
JP Morgan
Microsoft
NASDAQ Composite
Oil
S&P 500
USD-JPY
Walmart


Now to try our financial crystal ball hypothesis we will shift the sentiment scores so that we are using tomorrows news sentiments for guessing.

In [61]:
j = 0
for merged_data in merged_dataframes:
    stock = stocks_in_order[j]
    print(stock)
    data = pd.DataFrame()
    data['Date'] = merged_data['Date']
    # Create 'Day 1-10' columns
    for i in range(1, 11):
        # Change disney to appropriate stock name
        data[f'Day {i}'] = (merged_data[stock].shift(i) - merged_data[stock].shift(i-1)) / merged_data[stock].shift(i-1)
    # Create 'sentiment' column
    data['mean_score'] = merged_data['mean_score']
    data['mean_class'] = merged_data['mean_class']
    data['majority_vote_score'] = merged_data['majority_vote_score']
    data['median_score'] = merged_data['median_score']

    # Shift the mean_score, mean_class, majority_vote_score and median_score up by 1
    data['mean_score'] = data['mean_score'].shift(-1)
    data['mean_class'] = data['mean_class'].shift(-1)
    data['majority_vote_score'] = data['majority_vote_score'].shift(-1)
    data['median_score'] = data['median_score'].shift(-1)

    data.dropna(inplace=True)
    data.reset_index(drop=True, inplace=True)

    df = data.copy()

    price_columns = [f'Day {i}' for i in range(1, 11)]

    # Get the day 1 of the row below for next_return calculationrn
    df['next_return'] = df['Day 1'].shift(-1)

    # Drop rows where next_return is NaN (last row)
    df = df[:-1]

    # Create movement label (classification target)
    def get_movement_label(r):
        if r > 0.005:
            return 2
        elif r < -0.005:
            return 0
        else:
            return 1

    df['movement_label'] = df['next_return'].apply(get_movement_label)

    # Final columns: Day1-Day10, Sentiment, movement_label / next_return
    classification_df = df[price_columns + ['mean_score','mean_class','majority_vote_score','median_score', 'movement_label', 'Date']]
    regression_df = df[price_columns + ['mean_score','mean_class','majority_vote_score','median_score', 'next_return','Date']]

    classification_df.to_csv(f"LSTM Models/shifted_data/classification_data_returns_{stock}.csv", index=False)
    regression_df.to_csv(f"LSTM Models/shifted_data/regression_data_returns_{stock}.csv", index=False)

    j = j + 1

10Y Treasury
Apple
Coca-Cola
Disney
Dow Jones
EUR-USD
Exxon Mobil
Fed Funds Rate
Gold
Google
Intel
Johnson & Johnson
JP Morgan
Microsoft
NASDAQ Composite
Oil
S&P 500
USD-JPY
Walmart
